In [ ]:
# Importar librerías utilizadas para EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy.stats import chi2_contingency

In [ ]:
# Importar librerías utilizadas para partición de datos y entrenamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score, log_loss, brier_score_loss, accuracy_score, balanced_accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline

In [ ]:
# Importar librerías para el tratamiento de los artefactos
import joblib
import json
import os

In [ ]:
# Leer el fichero con el dataset
df_original = pd.read_csv('data/diabetes_dataset.csv')

# Crear una copia de trabajo del dataset
df = df_original.copy()

In [ ]:
# Mostrar la estructura del dataset
df.info()

In [ ]:
# Contar el número de valores nulos por columna
print('Nulos por columna:')
print(df.isnull().sum())

# Contar el número total de nulos
print('Total de valores nulos:', df.isnull().sum().sum())

# Contar el número de registros duplicados
print('Registros duplicados:', df.duplicated().sum())

In [ ]:
# Mostrar descriptivos de las variables categóricas
display(df.describe(include=['object']).T)

In [ ]:
# Analizar de forma específica la variable objetivo

# Frecuencias absolutas
print('Frecuencia de la variable objetivo (diabetes_stage):')
print(df['diabetes_stage'].value_counts())

# Frecuencias relativas (%)
print('\nPorcentaje de la variable objetivo (diabetes_stage):')
print((df['diabetes_stage'].value_counts(normalize=True) * 100).round(3))

# Tabla resumen
tabla_objetivo = pd.DataFrame({
    'Frecuencia': df['diabetes_stage'].value_counts(),
    'Porcentaje (%)': (df['diabetes_stage'].value_counts(normalize=True) * 100).round(3)
})

display(tabla_objetivo)

plt.figure(figsize=(8, 4))
sns.countplot(
    data=df,
    x='diabetes_stage',
    order=df['diabetes_stage'].value_counts().index,
    color='steelblue',
    edgecolor='black'
)
plt.title('Distribución de la variable objetivo: diabetes_stage', fontsize=13, fontweight='bold')
plt.xlabel(None)
plt.ylabel('Frecuencia')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar la distribución de cada una de las variables categóricas, excepto la variable objetivo
for col in [c for c in df.select_dtypes(include=['object']).columns if c != 'diabetes_stage']:
    plt.figure(figsize=(10, 4))
    plt.title(col, fontsize=14, fontweight='bold')
    sns.countplot(data=df, x=col, color='steelblue' , edgecolor='black')
    plt.xticks(rotation=45)
    plt.xlabel(None)
    plt.ylabel('Frecuencia')
    plt.show()

In [ ]:
# Convertir variables categóricas nominales a su tipo correcto
for col in ['gender', 'ethnicity', 'smoking_status', 'diabetes_stage', 'employment_status']:
    df[col] = df[col].astype('category')

# Convertir variables categóricas ordinales a su tipo correcto ordenado
df['education_level'] = pd.Categorical(
    df['education_level'], 
    categories=['No formal', 'Highschool', 'Graduate', 'Postgraduate'], 
    ordered=True
)

df['income_level'] = pd.Categorical(
    df['income_level'], 
    categories=['Low', 'Lower-Middle', 'Middle', 'Upper-Middle', 'High'], 
    ordered=True
)

In [ ]:
# Mostrar descriptivos de las variables numéricas
df.describe(include = ['number']).round(4).T

In [ ]:
# Mostrar la distribución de cada una de las variables numéricas
for col in df.select_dtypes(include = ['number']).columns:
    plt.figure(figsize=(10, 4))
    plt.title(col, fontsize=14, fontweight='bold')
    sns.histplot(data=df, x=col, color='steelblue' , edgecolor='black')
    plt.xlabel(None)
    plt.ylabel('Frecuencia')
    plt.show()

In [ ]:
# Distribución por clase de variables seleccinadas
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

cols = ['glucose_postprandial', 'glucose_fasting', 'hba1c', 'bmi']
clases = ['Gestational', 'No Diabetes', 'Pre-Diabetes', 'Type 1', 'Type 2']

for ax, col in zip(axes.flatten(), cols):
    for c in clases:
        subset = df[df['diabetes_stage'] == c]
        sns.kdeplot(data=subset, x=col, ax=ax, linewidth=1.8, label=c)
    
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Densidad")
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    if ax.get_legend() is not None:
        ax.get_legend().remove()


handles, labels = axes[0][0].get_legend_handles_labels()

fig.legend(handles, labels,
           loc='lower center',
           ncol=3,
           frameon=False)

plt.subplots_adjust(bottom=0.22)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

In [ ]:
# Convertir variables binarias codificadas como numéricas a variables categóricas
for col in ['family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'diagnosed_diabetes']:
    df[col] = df[col].astype('category')

In [ ]:
# Comprobar el resultado de las conversiones realizadas
df.info()

# Contar el número de valores nulos
print('Nulos por columna:', df.isnull().sum().sum())

# Contar el número de registros duplicados
print('Registros duplicados:', df.duplicated().sum())

df.describe(include = ['number']).round(4).T
df.describe(include = ['category']).T

In [ ]:
# Calcular la correlación entre las variables numéricas
corr_matrix = df.select_dtypes(include=['number']).corr()

# Mostrar la matriz de correlaciones
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación de variables numéricas', fontsize=14, fontweight='bold')
plt.show()

# Ordenar las correlaciones de mayor a menor (valor absoluto) en una lista
sol = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)).stack().reset_index())

sol.columns = ['Variable 1', 'Variable 2', 'Correlación']

sol['Abs_Corr'] = sol['Correlación'].abs()

correlaciones_ordenadas = sol.sort_values(by='Abs_Corr', ascending=False).reset_index(drop=True)

# Mostrar las 20 parejas de variables con mayor correlación
print('Parejas de variables con mayor correlación:')
display(correlaciones_ordenadas.head(20).drop(columns=['Abs_Corr']))

In [ ]:
# Definir la función para el cálculo de la V de Cramer entre varibles categóricas
def cramers_v (x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

In [ ]:
# Calcular la V de Cramer para todas las parejas de variables categóricas

# Variables categóricas
df_cat = df.select_dtypes(include=['category'])
cols = df_cat.columns
n = len(cols)

# Matriz vacía para los calculos
cramers_matrix = pd.DataFrame(np.zeros((n, n)), columns=cols, index=cols)

# Rellenar la matriz
for i in range(n):
    for j in range(i, n):
        val = cramers_v(df_cat.iloc[:, i], df_cat.iloc[:, j])
        cramers_matrix.iloc[i, j] = val
        cramers_matrix.iloc[j, i] = val

# Ordenar las V de Cramer de mayor a menor en una lista
sol_cat = (cramers_matrix.where(np.triu(np.ones(cramers_matrix.shape), k=1).astype(bool)).stack().reset_index())

sol_cat.columns = ['Variable 1', 'Variable 2', 'V de Cramer']

asociaciones_ordenadas = sol_cat.sort_values(by='V de Cramer', ascending=False).reset_index(drop=True)

# Mostrar las 20 parejas de variables con mayor asociación
print('Parejas de variables categóricas con mayor asociación:')
display(asociaciones_ordenadas.head(20))

In [ ]:
# Definir la función para calcular la razón de correlación entre variables categóricas y numéricas
def correlation_ratio (categories, measurements):
    categories = categories.astype(str)
    measurements = measurements.fillna(measurements.mean())
    
    fcat = categories.astype('category').cat.codes
    array = np.array(measurements)
    
    # Media global
    avg_y = np.mean(array)
    
    # Suma de cuadrados total
    ss_total = np.sum((array - avg_y)**2)
    
    # Suma de cuadrados explicada (entre grupos)
    ss_explained = 0
    for i in np.unique(fcat):
        y_group = array[fcat == i]
        ss_explained += len(y_group) * (np.mean(y_group) - avg_y)**2
            
    return np.sqrt(ss_explained / ss_total) if ss_total != 0 else 0

In [ ]:
# Variables categóricas y numéricas
df_cat = df.select_dtypes(include=['category'])
df_num = df.select_dtypes(include=['number'])

resultados = []

# Rellenar la lista comparando cada variables categórica con numérica
for col_cat in df_cat.columns:
    for col_num in df_num.columns:
        val = correlation_ratio(df_cat[col_cat], df_num[col_num])
        resultados.append([col_cat, col_num, val])

# Ordenar las ETA de mayor a menor en una lista
solucion = pd.DataFrame(resultados, columns=['Variable 1', 'Variable 2', 'ETA'])

etas_calculadas = solucion.sort_values(by='ETA', ascending=False).reset_index(drop=True)

## Mostrar las 20 parejas de variables con mayor asociación
print('Parejas Categórica-Numérica con mayor asociación:')
display(etas_calculadas.head(20))

In [ ]:
# Mostrar diagramas de cajas para las variables numéricas
df_num = df.select_dtypes(include=['number'])
columnas = df_num.columns
n_cols = 3
n_rows = math.ceil(len(columnas) / n_cols)

plt.figure(figsize=(15, 4 * n_rows))

# 3. Bucle para crear cada boxplot
for i, col in enumerate(columnas):
    ax = plt.subplot(n_rows, n_cols, i + 1)
    
    # showmeans=True ayuda a ver si la media se aleja mucho de la mediana por los outliers
    sns.boxplot(y=df_num[col], ax=ax, color='steelblue', fliersize=4, showmeans=True,
                meanprops={'marker':'^', 'markerfacecolor':'white', 'markeredgecolor':'black'})
    
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('')
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Diagramas de cajas de variables numéricas seleccionadas por relevancia clínica / asociación observada

variables_importantes_boxplot = [
    'hba1c',
    'glucose_fasting',
    'glucose_postprandial',
    'bmi',
    'diabetes_risk_score',
    'waist_to_hip_ratio'
]

n_cols = 3
n_rows = math.ceil(len(variables_importantes_boxplot) / n_cols)

plt.figure(figsize=(15, 4 * n_rows))

for i, col in enumerate(variables_importantes_boxplot):
    ax = plt.subplot(n_rows, n_cols, i + 1)

    sns.boxplot(
        y=df[col],
        ax=ax,
        color='steelblue',
        fliersize=4,
        showmeans=True,
        meanprops={'marker': '^', 'markerfacecolor': 'white', 'markeredgecolor': 'black'}
    )

    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('')
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Diagramas por clase de la variable objetivo para variables clave
n_cols = 2
n_rows = math.ceil(len(variables_importantes_boxplot) / n_cols)

plt.figure(figsize=(14, 5 * n_rows))

for i, col in enumerate(variables_importantes_boxplot):
    ax = plt.subplot(n_rows, n_cols, i + 1)

    sns.boxplot(
        data=df,
        x='diabetes_stage',
        y=col,
        ax=ax,
        color='steelblue',
        fliersize=3
    )

    ax.set_title(f'{col} por diabetes_stage', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Definir una semilla global para todo el procedimiento y garantizar reproducibilidad
SEED = 1434

In [ ]:
# Codificar los posibles valores de la variable objetivo
le = LabelEncoder()
df['diabetes_stage_cod'] = le.fit_transform(df['diabetes_stage'])

In [ ]:
# Separar el conjunto de Test (20%). 
# Se usa stratify sobre la variable objetivo para que se dividan los datos proporcionalmente a la distribución de las clases
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['diabetes_stage_cod'])

# Separar los conjuntos de Entrenamiento (60% total) y de Calibración (20% total)
df_train, df_calib = train_test_split(df_full_train, test_size=0.25, random_state=SEED, stratify=df_full_train['diabetes_stage_cod'])

# Separar la variable objetivo (diabetes_stage) del resto en los 3 conjuntos
y_train = df_train['diabetes_stage_cod'].values
y_calib = df_calib['diabetes_stage_cod'].values
y_test = df_test['diabetes_stage_cod'].values

# Borrar la variable objetivo (diabetes_stage y diabetes_stage_cod) de los conjuntos
cols_borrar = ['diabetes_stage', 'diabetes_stage_cod']

df_train = df_train.drop(columns=cols_borrar)
df_calib = df_calib.drop(columns=cols_borrar)
df_test = df_test.drop(columns=cols_borrar)

# Eliminar diagnosed_diabetes por fuga de información detectada en el EDA
cols_borrar = ['diagnosed_diabetes']

df_train = df_train.drop(columns=cols_borrar)
df_calib = df_calib.drop(columns=cols_borrar)
df_test = df_test.drop(columns=cols_borrar)


x_train = df_train.copy()
x_calib = df_calib.copy()
x_test  = df_test.copy()

In [ ]:
# Seleccionar variables a usar y transformarlas. 
# Las numéricas se estandarizarán y a las categóricas se le aplica One Hot Encoder

variables_num = df_train.select_dtypes(include=['number']).columns.tolist()
variables_cat = df_train.select_dtypes(include=['category']).columns.tolist()

# Crear el transformador para aplicar posteriormente a todas las variables según el tipo. 
# Se utiliza sparse_output=False para que guarde las columnas completas a la hora de realizar One Hot Encoder
# remainder='drop' elimina las variables no seleccionadas
transformador = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), variables_num), # Utiliza cuartiles y no media y desviación típica
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), variables_cat)
    ],
    remainder='drop'
)

In [ ]:
# Configuración general para el entrenamiento de todos los modelos
# Cross Validation de 5 rondas
estrategia_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

metricas = {
    'F1_Weighted': make_scorer(f1_score, average='weighted'),
    'F1_Macro': make_scorer(f1_score, average='macro'),
    'Balanced_Acc': make_scorer(balanced_accuracy_score),
    'Precision_Weighted': make_scorer(precision_score, average='weighted', zero_division=0),
    'Recall_Weighted': make_scorer(recall_score, average='weighted'),
    'LogLoss': 'neg_log_loss', 
    'Brier': make_scorer(brier_score_loss, response_method='predict_proba', greater_is_better=False)
}

In [ ]:
# Función para evaluar las métricas de los modelos
def evaluar_modelo(modelo, X, y, cv, metricas):
    resultados = cross_validate(
        modelo,
        X,
        y,
        cv=cv,
        scoring=metricas,
        return_train_score=True,
        n_jobs=-1
    )
    
    df_res = pd.DataFrame(resultados)
    
    if 'test_LogLoss' in df_res.columns:
        df_res['test_LogLoss'] = -df_res['test_LogLoss']
        df_res['train_LogLoss'] = -df_res['train_LogLoss']

    if 'test_Brier' in df_res.columns:
        df_res['test_Brier'] = -df_res['test_Brier']
        df_res['train_Brier'] = -df_res['train_Brier']

    mean = df_res.mean()
    std  = df_res.std()
    resumen = pd.DataFrame({'mean': mean, 'std': std})
    return resumen

In [ ]:
# Entranar un modelo sin ajustar para Logistic Regression con hiperparámetros comunes
lr_sin_ajustar = LogisticRegression(
    solver='lbfgs',
    multi_class='multinomial',
    penalty='l2',
    C=1.0,
    max_iter=2000,
    class_weight=None,
    random_state=SEED
)

pipe_lr_sin_ajustar = Pipeline([
    ('preprocess', transformador),
    ('model', lr_sin_ajustar)
])

resultado_lr_sin_ajustar = evaluar_modelo(pipe_lr_sin_ajustar, x_train, y_train, estrategia_cv, metricas)

In [ ]:
# Entranar un modelo sin ajustar para KNN con hiperparámetros comunes
knn_sin_ajustar = KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    metric='minkowski',
    p=2
)

pipe_knn_sin_ajustar = Pipeline([
    ('preprocess', transformador),
    ('model', knn_sin_ajustar)
])

resultado_knn_sin_ajustar = evaluar_modelo(pipe_knn_sin_ajustar, x_train, y_train, estrategia_cv, metricas)

In [ ]:
# Entrenar un modelo sin ajustar para Random Forest
rf_sin_ajustar = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight=None,
    random_state=SEED,
    n_jobs=-1
)


pipe_rf_sin_ajustar = Pipeline([
    ('preprocess', transformador),
    ('model', rf_sin_ajustar)
])

resultado_rf_sin_ajustar = evaluar_modelo(pipe_rf_sin_ajustar, x_train, y_train, estrategia_cv, metricas)

In [ ]:
# Entrenar un modelo sin ajustar para XGBoost
xgb_sin_ajustar = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    min_child_weight=1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective='multi:softprob',
    num_class=len(np.unique(y_train)),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1
)


pipe_xgb_sin_ajustar = Pipeline([
    ('preprocess', transformador),
    ('model', xgb_sin_ajustar)
])

resultado_xgb_sin_ajustar = evaluar_modelo(pipe_xgb_sin_ajustar, x_train, y_train, estrategia_cv, metricas)

In [ ]:
# Definir función general para el uso de Optuna para todos los modelos
def optuna_objective (trial, modelo, param, x, y, cv, scoring, transf):
    """
    trial            → Optuna trial
    modelo           → función que construye el modelo
    param            → espacio de hiperparámetros
    x, y             → datos
    cv               → estrategia de CV
    scoring          → scoring
    transf           → transformador para las variables en CV
    """
    
    params = param(trial)
    model = modelo(params)
    
    pipe = Pipeline([
        ('preprocess', transf),
        ('model', model)
    ])
    
    scores = cross_validate(
        pipe,
        x,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score='raise'
    )

    return scores['test_score'].mean()

In [ ]:
# Definir espacio de búsqueda de hiperparámentros para Logistic Regression
def param_lr(trial):
    return {
        'C': trial.suggest_float('C', 1e-4, 10.0, log=True),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', None])
    }

def modelo_lr(params):
    return LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        multi_class='multinomial',
        max_iter=2000,
        random_state=SEED,
        **params
    )

In [ ]:
# Definir espacio de búsqueda de hiperparámentros para KNN
def param_knn(trial):
    p = trial.suggest_categorical('p', [1, 2])  # 1=manhattan, 2=euclidean
    return {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 51, step=2),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'metric': 'minkowski',
        'p': p
    }

def modelo_knn(params):
    return KNeighborsClassifier(
        n_jobs=-1,
        **params
    )

In [ ]:
# Definir espacio de búsqueda de hiperparámentros para Random Forest
def param_rf(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=100),
        'max_depth': trial.suggest_categorical('max_depth', [None, 8, 10, 12, 14, 16]),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }

def modelo_rf(params):
    return RandomForestClassifier(
        criterion='gini',
        random_state=SEED,
        n_jobs=-1,
        **params
    )

In [ ]:
# Definir espacio de búsqueda de hiperparámentros para XGBoost
def param_xgb(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=100),
        'max_depth': trial.suggest_int('max_depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 3),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0)
    }

def modelo_xgb(params):
    return XGBClassifier(
        objective='multi:softprob',
        num_class=len(np.unique(y_train)),
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
        **params
    )

In [ ]:
# Definir mética para optimización con Optuma
metrica_optimizacion='f1_macro'
trials_optuna=50
trials_optuna_xgb=100

In [ ]:
# Entrenamiento y selección de modelo para Logistic Regression
sampler = optuna.samplers.TPESampler(seed=SEED)
study_lr = optuna.create_study(direction='maximize', sampler=sampler)

study_lr.enqueue_trial({
    'C': 1.0,
    'class_weight': None
})

study_lr.optimize(
    lambda trial: optuna_objective(
        trial,
        modelo_lr,
        param_lr,
        x_train,
        y_train,
        estrategia_cv,
        scoring=metrica_optimizacion,
        transf=transformador
    ),
    n_trials=trials_optuna
)

In [ ]:
# Entrenamiento y selección de modelo para KNN
sampler = optuna.samplers.TPESampler(seed=SEED)
study_knn = optuna.create_study(direction='maximize', sampler=sampler)

study_knn.enqueue_trial({
    'n_neighbors': 5,
    'weights': 'uniform',
    'p': 2
})

study_knn.optimize(
    lambda trial: optuna_objective(
        trial,
        modelo_knn,
        param_knn,
        x_train,
        y_train,
        estrategia_cv,
        scoring=metrica_optimizacion,
        transf=transformador
    ),
    n_trials=trials_optuna
)

In [ ]:
# Entrenamiento y selección de modelo para Random Forest
sampler = optuna.samplers.TPESampler(seed=SEED)
study_rf = optuna.create_study(direction='maximize', sampler=sampler)

study_rf.enqueue_trial({
    'n_estimators': 100,
    'max_depth': None,
    'min_samples_leaf': 1,
    'class_weight': None,
    'min_samples_split': 2,
    'max_features': 'sqrt'
})

study_rf.optimize(
    lambda trial: optuna_objective(
        trial,
        modelo_rf,
        param_rf,
        x_train,
        y_train,
        estrategia_cv,
        scoring=metrica_optimizacion,
        transf=transformador
    ),
    n_trials=trials_optuna
)

In [ ]:
# Entrenamiento y selección de modelo para XGBoost
sampler = optuna.samplers.TPESampler(seed=SEED)
study_xgb = optuna.create_study(direction='maximize', sampler=sampler)

study_xgb.enqueue_trial({
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.1,
    'min_child_weight': 1,
    'subsample': 1.0,
    'colsample_bytree': 1.0,
    'reg_lambda': 1.0,
    'reg_alpha': 0.0
})

study_xgb.optimize(
    lambda trial: optuna_objective(
        trial,
        modelo_xgb,
        param_xgb,
        x_train,
        y_train,
        estrategia_cv,
        scoring=metrica_optimizacion,
        transf=transformador
    ),
    n_trials=trials_optuna_xgb
)

In [ ]:
# Entrenar los modelos con los mejores hiperparámetros obtenidos para cada uno de ellos
lr_ajustado = modelo_lr(study_lr.best_params)
pipe_lr_ajustado = Pipeline([
    ('preprocess', transformador),
    ('model', lr_ajustado)
])
pipe_lr_ajustado.fit(x_train, y_train)

knn_ajustado = modelo_knn(study_knn.best_params)
pipe_knn_ajustado = Pipeline([
    ('preprocess', transformador),
    ('model', knn_ajustado)
])
pipe_knn_ajustado.fit(x_train, y_train)

rf_ajustado = modelo_rf(study_rf.best_params)
pipe_rf_ajustado = Pipeline([
    ('preprocess', transformador),
    ('model', rf_ajustado)
])
pipe_rf_ajustado.fit(x_train, y_train)

xgb_ajustado = modelo_xgb(study_xgb.best_params)
pipe_xgb_ajustado = Pipeline([
    ('preprocess', transformador),
    ('model', xgb_ajustado)
])
pipe_xgb_ajustado.fit(x_train, y_train)

In [ ]:
resultado_lr_ajustado = evaluar_modelo(pipe_lr_ajustado, x_train, y_train, estrategia_cv, metricas)

resultado_knn_ajustado = evaluar_modelo(pipe_knn_ajustado, x_train, y_train, estrategia_cv, metricas)

resultado_rf_ajustado = evaluar_modelo(pipe_rf_ajustado, x_train, y_train, estrategia_cv, metricas)

resultado_xgb_ajustado = evaluar_modelo(pipe_xgb_ajustado, x_train, y_train, estrategia_cv, metricas)

In [ ]:
# Función para conseguir las métricas de un modelo
def fila_resultados(nombre_modelo, tipo, resumen):
    def pull(name):
        return resumen.loc[name, 'mean'], resumen.loc[name, 'std']

    return {
        'Modelo': nombre_modelo,
        'Tipo': tipo,

        # VALIDACIÓN (CV)
        'F1_Macro_mean': pull('test_F1_Macro')[0],
        'F1_Macro_std':  pull('test_F1_Macro')[1],
        'Balanced_Acc_mean': pull('test_Balanced_Acc')[0],
        'Balanced_Acc_std':  pull('test_Balanced_Acc')[1],
        'F1_Weighted_mean': pull('test_F1_Weighted')[0],
        'F1_Weighted_std':  pull('test_F1_Weighted')[1],
        'LogLoss_mean': pull('test_LogLoss')[0],
        'LogLoss_std':  pull('test_LogLoss')[1],
        'Brier_mean': pull('test_Brier')[0],
        'Brier_std':  pull('test_Brier')[1],

        # ENTRENAMIENTO (CV)
        'train_F1_Macro_mean': pull('train_F1_Macro')[0],
        'train_F1_Macro_std':  pull('train_F1_Macro')[1],
    }

In [ ]:
filas = []

# ---- Sin ajustar ----
filas.append(fila_resultados('Logistic Regression', 'Sin ajustar', resultado_lr_sin_ajustar))
filas.append(fila_resultados('KNN', 'Sin ajustar', resultado_knn_sin_ajustar))
filas.append(fila_resultados('Random Forest', 'Sin ajustar', resultado_rf_sin_ajustar))
filas.append(fila_resultados('XGBoost', 'Sin ajustar', resultado_xgb_sin_ajustar))

# ---- Ajustados (Optuna) ----
filas.append(fila_resultados('Logistic Regression', 'Ajustado', resultado_lr_ajustado))
filas.append(fila_resultados('KNN', 'Ajustado', resultado_knn_ajustado))
filas.append(fila_resultados('Random Forest', 'Ajustado', resultado_rf_ajustado))
filas.append(fila_resultados('XGBoost', 'Ajustado', resultado_xgb_ajustado))

df_resultados = pd.DataFrame(filas)

cols_num = [c for c in df_resultados.columns if c.endswith('_mean') or c.endswith('_std')]
df_resultados[cols_num] = df_resultados[cols_num].round(4)

df_resultados

In [ ]:
# Entrenar y seleccionar modelo mediante GridSearch para Random Forest
param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 8, 12],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

def modelo_grid_rf():
    return RandomForestClassifier(
        criterion='gini',
        random_state=SEED,
        n_jobs=-1,
    )

pipe_grid_rf = Pipeline([
    ('preprocess', transformador),
    ('model', modelo_grid_rf())
])

# Definición del Grid
grid_rf = GridSearchCV(
    estimator=pipe_grid_rf,  
    param_grid=param_grid_rf,
    cv=estrategia_cv,
    scoring=metrica_optimizacion,
    n_jobs=-1
)

# Entrenamiento
grid_rf.fit(x_train, y_train)

In [ ]:
# Entrenar y seleccionar modelo mediante GridSearch para XGBoost
param_grid_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [4, 5, 6],
    'model__learning_rate': [0.03, 0.1, 0.2],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
}

def modelo_grid_xgb():
    return XGBClassifier(
        objective='multi:softprob',
        num_class=len(np.unique(y_train)),
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1
    )


pipe_grid_xgb = Pipeline([
    ('preprocess', transformador),
    ('model', modelo_grid_xgb())
])

# Definición del Grid
grid_xgb = GridSearchCV(
    estimator=pipe_grid_xgb,
    param_grid=param_grid_xgb,
    cv=estrategia_cv,
    scoring=metrica_optimizacion,
    n_jobs=-1
)

# Entrenamiento
grid_xgb.fit(x_train, y_train)

In [ ]:
# Extraer mejores parámetros obtenidos con los grids
mejores_params_rf = grid_rf.best_params_
mejores_params_xgb = grid_xgb.best_params_

# Limpiar nombres de las variables que nos devuelve el grid
mejores_params_rf = {k.replace('model__', ''): v for k, v in mejores_params_rf.items()}
mejores_params_xgb = {k.replace('model__', ''): v for k, v in mejores_params_xgb.items()}

# Construir los modelos
rf_ajustado_grid = modelo_rf(mejores_params_rf)
xgb_ajustado_grid = modelo_xgb(mejores_params_xgb)

# Pipelines
pipe_rf_ajustado_grid = Pipeline([
    ('preprocess', transformador),
    ('model', rf_ajustado_grid)
])

pipe_xgb_ajustado_grid = Pipeline([
    ('preprocess', transformador),
    ('model', xgb_ajustado_grid)
])

# Entrenamiento del mejor modelo
pipe_rf_ajustado_grid.fit(x_train, y_train)
pipe_xgb_ajustado_grid.fit(x_train, y_train)

# Evaluación
resultado_grid_rf = evaluar_modelo(pipe_rf_ajustado_grid, x_train, y_train, estrategia_cv, metricas)
resultado_grid_xgb = evaluar_modelo(pipe_xgb_ajustado_grid, x_train, y_train, estrategia_cv, metricas)

In [ ]:
filas_con_grid = filas.copy()

filas_con_grid.append(fila_resultados('Random Forest', 'Grid', resultado_grid_rf))
filas_con_grid.append(fila_resultados('XGBoost', 'Grid', resultado_grid_xgb))

df_resultados_grid = pd.DataFrame(filas_con_grid)

cols_num_grid = [c for c in df_resultados_grid.columns if c.endswith('_mean') or c.endswith('_std')]
df_resultados_grid[cols_num_grid] = df_resultados_grid[cols_num_grid].round(4)

df_resultados_grid

In [ ]:
# Matriz de confusión para Random Forest ajustado con los datos que no entran en el los Fold de CV (Out of fold OOF)
# Predicciones OOF
y_pred_oof_rf = cross_val_predict(
    pipe_rf_ajustado,
    x_train,
    y_train,
    cv=estrategia_cv,
    method='predict',
    n_jobs=-1
)

# Matriz de confusión OOF
labels = np.arange(len(le.classes_))
cm_oof_rf = confusion_matrix(
    y_train,
    y_pred_oof_rf,
    labels=labels
)

# Normalización por filas (recall por clase)
cm_oof_rf_norm = cm_oof_rf / np.maximum(cm_oof_rf.sum(axis=1, keepdims=True), 1)

# Mostrar matriz normalizada
disp_rf = ConfusionMatrixDisplay(
    confusion_matrix=cm_oof_rf_norm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(7, 6))
disp_rf.plot(
    ax=ax,
    cmap='Blues',
    values_format='.2f',
    xticks_rotation=45
)

plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
#plt.title('Matriz de confusión (OOF, normalizada por clase)')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusión para XGBoost ajustado con los datos que no entran en el los Fold de CV (Out of fold OOF)
# Predicciones OOF
y_pred_oof_xgb = cross_val_predict(
    pipe_xgb_ajustado,
    x_train,
    y_train,
    cv=estrategia_cv,
    method='predict',
    n_jobs=-1
)

# Matriz de confusión OOF
labels = np.arange(len(le.classes_))
cm_oof_xgb = confusion_matrix(
    y_train,
    y_pred_oof_xgb,
    labels=labels
)

# Normalización por filas (recall por clase)
cm_oof_xgb_norm = cm_oof_xgb / np.maximum(cm_oof_xgb.sum(axis=1, keepdims=True), 1)

# Mostrar matriz normalizada
disp_xgb = ConfusionMatrixDisplay(
    confusion_matrix=cm_oof_xgb_norm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(7, 6))
disp_xgb.plot(
    ax=ax,
    cmap='Blues',
    values_format='.2f',
    xticks_rotation=45
)

plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
#plt.title('Matriz de confusión (OOF, normalizada por clase)')
plt.tight_layout()
plt.show()

In [ ]:
# Entrenar el modelo final con todos los datos de train sin validación cruzada (XGBoost)
pipe_final_xgb = Pipeline([
    ('preprocess', transformador),
    ('model', modelo_xgb(study_xgb.best_params))
])
pipe_final_xgb.fit(x_train, y_train)

In [ ]:
# Cálculo de probabilidades predichas sobre el conjunto de test
y_prob_test_xgb = pipe_final_xgb.predict_proba(x_test)

# Predicciones sobre el conjunto de test
y_pred_test_xgb = pipe_final_xgb.predict(x_test)

In [ ]:
# Calcular las métricas sobre el conjunto de TEST
classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)

brier_xgb = np.mean(np.sum((y_prob_test_xgb - y_test_bin)**2, axis=1))
resultados_test_xgb = {
    'Modelo': 'XGBoost',
    'Tipo': 'Test',

    'F1_Macro': f1_score(y_test, y_pred_test_xgb, average='macro'),
    'Balanced_Acc': balanced_accuracy_score(y_test, y_pred_test_xgb),
    'F1_Weighted': f1_score(y_test, y_pred_test_xgb, average='weighted'),
    'LogLoss': log_loss(y_test, y_prob_test_xgb), 
    'Brier': brier_xgb
}

df_metricas_test = pd.DataFrame([resultados_test_xgb])

for col in ['F1_Macro', 'Balanced_Acc', 'F1_Weighted', 'LogLoss', 'Brier']:
    df_metricas_test[col] = df_metricas_test[col].map('{:.4f}'.format)

print(df_metricas_test.to_string(index=False))

In [ ]:
# Matriz de confusión sobre conjunto de TEST para XGBoost
labels = np.arange(len(le.classes_))
cm_test_xgb = confusion_matrix(
    y_test,
    y_pred_test_xgb,
    labels=labels
)

# Normalización por filas (recall por clase)
cm_test_norm_xgb = cm_test_xgb / np.maximum(cm_test_xgb.sum(axis=1, keepdims=True), 1)

# Mostrar matriz normalizada
disp_xgb = ConfusionMatrixDisplay(
    confusion_matrix=cm_test_norm_xgb,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(7, 6))
disp_xgb.plot(
    ax=ax,
    cmap='Blues',
    values_format='.2f',
    xticks_rotation=45
)

plt.xlabel('Clase predicha')
plt.ylabel('Clase real')
#plt.title('Matriz de confusión (conjunto de prueba, normalizada por clase)')

plt.tight_layout()
plt.show()

In [ ]:
# Crear carpeta de artefactos si no existe
os.makedirs('artefactos', exist_ok=True)

In [ ]:
# Guardar el modelo final entrenado
joblib.dump(
    {
        'modelo': pipe_final_xgb,
        'tipo_modelo': 'XGBoost'
    },
    'artefactos/pipe_final.joblib'
)

In [ ]:
# Guardar el label encoder generado
joblib.dump(le, 'artefactos/label_encoder.joblib')

In [ ]:
# Guardar las matrices de confusión
# Matriz OOF
joblib.dump(
    {
        'cm_oof': cm_oof_xgb,
        'cm_oof_norm': cm_oof_xgb_norm,
        'y_pred_oof': y_pred_oof_xgb
    },
    'artefactos/matriz_confusion_oof_final.joblib'
)

# Matriz TEST
joblib.dump(
    {
        'cm_test': cm_test_xgb,
        'cm_test_norm': cm_test_norm_xgb,
        'y_pred_test': y_pred_test_xgb
    },
    'artefactos/matriz_confusion_test_final.joblib'
)

In [ ]:
# Guardar tabla de métricas (CV)
joblib.dump(df_resultados_grid, 'artefactos/resultados_modelos_cv.joblib')

In [ ]:
# Guardar tabla de métricas (TEST XGBoost)
joblib.dump(df_metricas_test, 'artefactos/resultados_final_test.joblib')

In [ ]:
# Guardar los dataset de datos particionados
joblib.dump(
    {
        'x_train': x_train,
        'y_train': y_train,
        'x_calib': x_calib,
        'y_calib': y_calib,
        'x_test': x_test,
        'y_test': y_test,
    },
    'artefactos/datasets_splits.joblib'
)

In [ ]:
parametros = []

# Paráemtros globales
parametros.append({
    'Tipo': 'Meta',
    'Configuracion': {
        'seed': SEED,
        'cv_folds': estrategia_cv.n_splits,
        'shuffle_cv': estrategia_cv.shuffle,
        'metrica_optimizacion': metrica_optimizacion
    }
})

# Modelos sin ajustar
parametros.append({
    'Modelo': 'Logistic Regression',
    'Tipo': 'Sin ajustar',
    'Parametros': {k: (None if isinstance(v, float) and np.isnan(v) else v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) 
                   for k, v in pipe_lr_sin_ajustar.named_steps['model'].get_params().items()}
})

parametros.append({
    'Modelo': 'KNN',
    'Tipo': 'Sin ajustar',
    'Parametros': {k: (None if isinstance(v, float) and np.isnan(v) else v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) 
                   for k, v in pipe_knn_sin_ajustar.named_steps['model'].get_params().items()}
})

parametros.append({
    'Modelo': 'Random Forest',
    'Tipo': 'Sin ajustar',
    'Parametros': {k: (None if isinstance(v, float) and np.isnan(v) else v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) 
                   for k, v in pipe_rf_sin_ajustar.named_steps['model'].get_params().items()}
})

parametros.append({
    'Modelo': 'XGBoost',
    'Tipo': 'Sin ajustar',
    'Parametros': {k: (None if isinstance(v, float) and np.isnan(v) else v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) 
                   for k, v in pipe_xgb_sin_ajustar.named_steps['model'].get_params().items()}
})

# Modelos ajustados con Optuna
parametros.append({
    'Modelo': 'Logistic Regression',
    'Tipo': 'Ajustado',
    'Parametros': study_lr.best_params
})

parametros.append({
    'Modelo': 'KNN',
    'Tipo': 'Ajustado',
    'Parametros': study_knn.best_params
})

parametros.append({
    'Modelo': 'Random Forest',
    'Tipo': 'Ajustado',
    'Parametros': study_rf.best_params
})

parametros.append({
    'Modelo': 'XGBoost',
    'Tipo': 'Ajustado',
    'Parametros': study_xgb.best_params
})

# Modelos ajustados con Grid
parametros.append({
    'Modelo': 'Random Forest',
    'Tipo': 'Grid',
    'Parametros': {k.replace('model__', ''): v for k, v in grid_rf.best_params_.items()}
})

parametros.append({
    'Modelo': 'XGBoost',
    'Tipo': 'Grid',
    'Parametros': {k.replace('model__', ''): v for k, v in grid_xgb.best_params_.items()}
})

In [ ]:
# Guardar en un fichero JSon los hiperparámetros utilizados para entrenar los modelos sin ajustar y ajustados
with open('artefactos/parametros_modelos.json', 'w') as f:
    json.dump(parametros, f, indent=2)